# Face Presentation Attack Detection — Midterm Lượt 1, 2 và 3

**CASIA-FASD temporary benchmark · E0 MobileNetV3 · E1 CDCN + pseudo-depth**

Notebook này là runbook thực nghiệm hoàn chỉnh cho báo cáo giữa kỳ. Nó được thiết kế để:

- chạy tuần tự từ trên xuống dưới trên Google Colab;
- không huấn luyện lại hoặc retune E0 đã khóa;
- tạo queue pseudo-depth có thể resume;
- chặn E1 nếu depth QA chưa hợp lệ;
- chỉ mở test E1 sau khi checkpoint và threshold validation đã được khóa;
- lưu run, báo cáo và checkpoint quan trọng trên Google Drive.

> **Runtime khuyến nghị:** A100 khi chạy 3DDFA/E1. Các phần chuẩn bị dataset, manifest và kiểm tra protocol có thể chạy CPU.
> Không commit dataset, frame, pseudo-depth hoặc checkpoint lên GitHub.


## Cách sử dụng

1. Chạy lần lượt từng cell, không dùng **Run all** qua các cổng kiểm duyệt thủ công.
2. Ở cổng 3A, xem ảnh `casia-depth-smoke.png`, rồi mới đổi `SMOKE_APPROVED = True`.
3. Ở cổng 3C, xem loss và predicted-depth cases, rồi mới đổi `E1_SMOKE_APPROVED = True`.
4. Cell locked test mặc định không chạy. Chỉ đổi `RUN_LOCKED_TEST = True` đúng một lần sau khi E1 đã được chốt.
5. Nếu runtime mất kết nối, chạy lại từ đầu. Cell queue sẽ khôi phục các live depth map đã checkpoint lên Drive.


## 0. Môi trường, Git revision và dataset


In [ ]:
#@title 0.1 Mount Google Drive và khai báo đường dẫn
from google.colab import drive
drive.mount("/content/drive")

import importlib
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image as DisplayImage, display

REPO_URL = "https://github.com/Violetta147/face-recognition-depth-pad.git"
REPO_ROOT = Path("/content/face-recognition-depth-pad")
DATA_ROOT = Path("/content/datasets/casia-fasd")
DRIVE_ROOT = Path("/content/drive/MyDrive/face-pad")
RUNS_DIR = DRIVE_ROOT / "runs"
REPORTS_DIR = DRIVE_ROOT / "reports"
DEPTH_BACKUP = DRIVE_ROOT / "depth-checkpoints" / "casia-fasd"

MANIFEST = REPO_ROOT / "data/manifests/casia_fasd_debug.csv"
DEPTH_ROOT = DATA_ROOT / "depth"
DEPTH_LEDGER = DEPTH_ROOT / "depth_status.csv"
DEPTH_PENDING = DEPTH_ROOT / "3ddfa_pending.csv"
DEPTH_FAILURES = DEPTH_ROOT / "3ddfa_failures.csv"
DEPTH_MANIFEST = REPO_ROOT / "data/manifests/casia_fasd_debug_with_depth.csv"
THREEDDFA_ROOT = Path("/content/3DDFA_V2")
THREEDDFA_COMMIT = "1b6c67601abffc1e9f248b291708aef0e43b55ae"

for directory in (RUNS_DIR, REPORTS_DIR, DEPTH_BACKUP):
    directory.mkdir(parents=True, exist_ok=True)

def run(command, *, cwd=None):
    command = [str(item) for item in command]
    print("$", " ".join(command))
    subprocess.run(command, cwd=str(cwd) if cwd else None, check=True)

print("Drive root:", DRIVE_ROOT)
print("Runs:", RUNS_DIR)
print("Reports:", REPORTS_DIR)


In [ ]:
#@title 0.2 Clone/cập nhật repository và cài package
if (REPO_ROOT / ".git").is_dir():
    run(["git", "-C", REPO_ROOT, "pull", "--ff-only"])
else:
    run(["git", "clone", REPO_URL, REPO_ROOT])

run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], cwd=REPO_ROOT)

# Editable install trên một số runtime Colab không được kernel nhận ngay.
source_root = REPO_ROOT / "src"
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))
importlib.invalidate_caches()

import deepface_pad

git_revision = subprocess.check_output(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True
).strip()
print("Git revision:", git_revision)
print("deepface_pad:", deepface_pad.__file__)

required = [
    REPO_ROOT / "scripts/run_3ddfa_worker.py",
    REPO_ROOT / "scripts/score_checkpoint.py",
    REPO_ROOT / "scripts/visualize_depth_cases.py",
    REPO_ROOT / "configs/casia_e1_smoke.yaml",
    REPO_ROOT / "configs/casia_e1_cdcn.yaml",
]
missing = [str(path) for path in required if not path.is_file()]
assert not missing, f"Repository thiếu file: {missing}"


In [ ]:
#@title 0.3 Unit tests và GPU
import torch

run([sys.executable, "-m", "pytest", "-q"], cwd=REPO_ROOT)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


### 0.4 Chuẩn bị CASIA-FASD

Cell dưới đây dùng nguồn đầu tiên tìm thấy:

1. dataset đã có tại `/content/datasets/casia-fasd`;
2. thư mục Drive `MyDrive/face-pad/datasets/casia-fasd`;
3. file Drive `MyDrive/face-pad/casia-fasd.zip`;
4. tải public dataset `immada/casia-fasd` bằng KaggleHub.

Dataset luôn được chuẩn hóa về đường dẫn `/content/datasets/casia-fasd` mà config sử dụng.


In [ ]:
#@title 0.4 Chuẩn bị dataset (không cần sửa nếu đường dẫn chuẩn đã tồn tại)
def looks_like_casia(path: Path) -> bool:
    return (path / "train" / "live").is_dir() and (path / "test" / "spoof").is_dir()

def find_casia_root(base: Path) -> Path | None:
    if looks_like_casia(base):
        return base
    if base.exists():
        for candidate in base.rglob("train"):
            root = candidate.parent
            if looks_like_casia(root):
                return root
    return None

DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)

if not looks_like_casia(DATA_ROOT):
    drive_dataset = DRIVE_ROOT / "datasets" / "casia-fasd"
    dataset_zip = DRIVE_ROOT / "casia-fasd.zip"

    if looks_like_casia(drive_dataset):
        print("Copying dataset từ Drive vào local runtime để train nhanh hơn...")
        shutil.copytree(drive_dataset, DATA_ROOT, dirs_exist_ok=True)
    elif dataset_zip.is_file():
        print("Extracting:", dataset_zip)
        shutil.unpack_archive(str(dataset_zip), str(DATA_ROOT.parent))
        discovered = find_casia_root(DATA_ROOT.parent)
        if discovered and discovered.resolve() != DATA_ROOT.resolve():
            if DATA_ROOT.exists():
                raise RuntimeError(f"Đường dẫn đích đã tồn tại nhưng sai cấu trúc: {DATA_ROOT}")
            os.symlink(discovered, DATA_ROOT, target_is_directory=True)
    else:
        run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
        import kagglehub

        downloaded = Path(kagglehub.dataset_download("immada/casia-fasd"))
        discovered = find_casia_root(downloaded)
        if discovered is None:
            raise RuntimeError(f"Không tìm thấy train/test trong Kaggle download: {downloaded}")
        os.symlink(discovered, DATA_ROOT, target_is_directory=True)

assert looks_like_casia(DATA_ROOT), f"CASIA-FASD sai cấu trúc: {DATA_ROOT}"
print("Dataset root:", DATA_ROOT)
print("Train live:", DATA_ROOT / "train/live")
print("Train spoof:", DATA_ROOT / "train/spoof")
print("Test live:", DATA_ROOT / "test/live")
print("Test spoof:", DATA_ROOT / "test/spoof")


## 1. Lượt 1 — Dataset batch và protocol verification


In [ ]:
#@title 1.1 Tạo manifest và chạy validator subject-disjoint
run(
    [
        sys.executable,
        "scripts/prepare_casia_fasd.py",
        "--data-root", DATA_ROOT,
        "--output-dir", "data/manifests",
        "--frames-per-video", "20",
    ],
    cwd=REPO_ROOT,
)

run(
    [
        sys.executable,
        "scripts/validate_manifest.py",
        MANIFEST,
        "--data-root", DATA_ROOT,
        "--subject-disjoint",
    ],
    cwd=REPO_ROOT,
)


In [ ]:
#@title 1.2 Thống kê protocol và kiểm tra leakage độc lập
manifest = pd.read_csv(MANIFEST, keep_default_na=False, dtype={"sample_id": str})

summary = (
    manifest.groupby(["split", "label"])
    .agg(
        samples=("sample_id", "size"),
        videos=("video_id", "nunique"),
        subjects=("subject_id", "nunique"),
    )
)
display(summary)

assert len(manifest) == 12_000, f"Expected 12000 samples, found {len(manifest)}"
assert manifest["video_id"].nunique() == 600
assert manifest["subject_id"].nunique() == 50
assert manifest["sample_id"].is_unique
assert manifest.groupby("video_id")["split"].nunique().max() == 1
assert manifest.groupby("subject_id")["split"].nunique().max() == 1

print("✓ 12,000 frames / 600 videos / 50 subjects")
print("✓ Không trùng sample_id")
print("✓ Không video leakage")
print("✓ Không subject leakage")


In [ ]:
#@title 1.3 Tạo và hiển thị batch validation
L1_BATCH = REPORTS_DIR / "casia-val-batch.png"
run(
    [
        sys.executable,
        "scripts/visualize_manifest_batch.py",
        MANIFEST,
        "--data-root", DATA_ROOT,
        "--split", "val",
        "--count", "16",
        "--output", L1_BATCH,
    ],
    cwd=REPO_ROOT,
)
display(DisplayImage(filename=str(L1_BATCH), width=1100))
print("Saved:", L1_BATCH)


### Cổng Lượt 1

Kiểm tra bằng mắt rằng ảnh `LIVE` là bona fide, ảnh `ATTACK` là spoof và không có ảnh hỏng/nhãn đảo. Nếu đúng, Lượt 1 cho CASIA giữa kỳ hoàn tất.


## 2. Lượt 2 — E0 frozen result, không chạy lại/retune


E0 được giữ nguyên từ run:

`CASIA_E0_BCE_5E_seed42_20260920T082941Z`

Threshold `0.9900876432657242` được chọn trên validation trước khi mở test. Notebook này **không huấn luyện lại E0 và không chọn lại threshold**.


In [ ]:
#@title 2.1 Xác minh artifact E0 đã khóa
E0_RUN = RUNS_DIR / "CASIA_E0_BCE_5E_seed42_20260920T082941Z"
E0_TEST_METRICS = {
    "threshold": 0.9900876432657242,
    "apcer": 0.0,
    "bpcer": 0.1111111111111111,
    "acer": 0.05555555555555555,
    "eer": 0.024074074074074074,
    "auc": 0.9979835390946502,
}

required_e0 = ["config.yaml", "best.ckpt", "threshold.json", "metrics.json"]
missing_e0 = [name for name in required_e0 if not (E0_RUN / name).is_file()]
assert not missing_e0, f"E0 run thiếu artifact trên Drive: {missing_e0}"

threshold_payload = json.loads((E0_RUN / "threshold.json").read_text())
assert abs(float(threshold_payload["threshold"]) - E0_TEST_METRICS["threshold"]) < 1e-12
assert threshold_payload.get("source") == "validation"

print("Frozen E0 run:", E0_RUN)
display(pd.DataFrame([E0_TEST_METRICS], index=["E0 locked test"]))


In [ ]:
#@title 2.2 Xem training log và error analysis E0 đã có (không score lại test)
display(pd.read_csv(E0_RUN / "train_log.csv"))

if (E0_RUN / "test_scores.csv").is_file():
    e0_scores = pd.read_csv(E0_RUN / "test_scores.csv").reset_index(drop=True)
    metadata = (
        manifest[["video_id", "subject_id", "label", "attack_type", "quality"]]
        .drop_duplicates("video_id")
        .reset_index(drop=True)
    )
    e0_analysis = e0_scores.merge(metadata, on=["video_id", "label"], how="left")
    e0_analysis["prediction"] = (
        e0_analysis["score"] >= E0_TEST_METRICS["threshold"]
    ).astype(int)
    e0_analysis["correct"] = e0_analysis["prediction"] == e0_analysis["label"]
    display(
        e0_analysis.groupby(["label", "attack_type", "quality"], dropna=False).agg(
            videos=("video_id", "count"),
            errors=("correct", lambda values: (~values).sum()),
            mean_score=("score", "mean"),
            min_score=("score", "min"),
            max_score=("score", "max"),
        )
    )
else:
    print("test_scores.csv không có trên Drive; dùng số E0 frozen đã ghi ở trên, không score lại.")


### Kết luận E0 frozen

| Model | Epoch | Threshold source | APCER | BPCER | ACER | EER | AUC |
|---|---:|---|---:|---:|---:|---:|---:|
| MobileNetV3 Small | 5 | Validation | 0.00% | 11.11% | 5.56% | 2.41% | 99.80% |

Kết quả này chỉ là benchmark tạm thời trên CASIA-FASD; không tuyên bố khả năng tổng quát hóa cross-dataset.


## 3. Lượt 3 — Pseudo-depth và CDCN E1


### 3A. 3DDFA smoke test

Queue dùng zero target cho attack và 3DDFA V2 cho bona fide. Failure bona fide được ghi rõ; tuyệt đối không thay failure bằng zero target.

Các live depth map hoàn tất được checkpoint định kỳ lên Drive. Attack-zero target được tái tạo nhanh khi resume nên không cần sao lưu 9.000 file zero lên Drive.


In [ ]:
#@title 3A.1 Khôi phục depth checkpoint từ Drive (nếu có)
def restore_depth_checkpoint() -> None:
    if not (DEPTH_BACKUP / "depth_status.csv").is_file():
        print("Không có depth checkpoint cũ trên Drive.")
        return
    DEPTH_ROOT.mkdir(parents=True, exist_ok=True)
    restored = 0
    for source in DEPTH_BACKUP.iterdir():
        destination = DEPTH_ROOT / source.name
        if source.is_file() and (
            source.suffix == ".npy" or
            source.name.endswith(".csv") or
            source.name.endswith(".json")
        ):
            if not destination.exists() or source.stat().st_size != destination.stat().st_size:
                shutil.copy2(source, destination)
                restored += 1
    print(f"Restored {restored} checkpoint files từ {DEPTH_BACKUP}")

def save_depth_checkpoint() -> None:
    if not DEPTH_LEDGER.is_file():
        return
    DEPTH_BACKUP.mkdir(parents=True, exist_ok=True)
    ledger = pd.read_csv(DEPTH_LEDGER, keep_default_na=False, dtype={"sample_id": str})
    complete_live = ledger[
        (ledger["label"] == 1) & (ledger["status"] == "complete_bona_fide")
    ]
    copied = 0
    for row in complete_live.itertuples(index=False):
        source = Path(str(row.output_path))
        destination = DEPTH_BACKUP / source.name
        if source.is_file() and (
            not destination.exists() or source.stat().st_size != destination.stat().st_size
        ):
            shutil.copy2(source, destination)
            copied += 1
    for source in DEPTH_ROOT.iterdir():
        if source.is_file() and (source.name.endswith(".csv") or source.name.endswith(".json")):
            shutil.copy2(source, DEPTH_BACKUP / source.name)
    print(f"Checkpoint: {len(complete_live)} live maps ({copied} mới) → {DEPTH_BACKUP}")

restore_depth_checkpoint()


In [ ]:
#@title 3A.2 Tạo/resume queue pseudo-depth
prepare_queue_command = [
    sys.executable,
    "scripts/generate_depth.py",
    MANIFEST,
    "--data-root", DATA_ROOT,
    "--output-root", DEPTH_ROOT,
]

# Chỉ truyền failure report khi file đã tồn tại từ một worker trước đó.
if DEPTH_FAILURES.is_file():
    prepare_queue_command.extend([
        "--failure-report",
        DEPTH_FAILURES,
    ])

run(prepare_queue_command, cwd=REPO_ROOT)

depth_status = pd.read_csv(
    DEPTH_LEDGER,
    keep_default_na=False,
    dtype={"sample_id": str},
)

display(
    depth_status.groupby(["label", "status"])
    .size()
    .rename("samples")
    .to_frame()
)

print("Pending queue:", len(pd.read_csv(DEPTH_PENDING)))

In [ ]:
#@title 3A.3 Cài và build official 3DDFA V2 ở commit cố định
run([
    sys.executable, "-m", "pip", "install", "-q",
    "cython", "imageio", "imageio-ffmpeg", "pyyaml", "tqdm",
    "scikit-image", "scipy", "opencv-python-headless<5",
])

if (THREEDDFA_ROOT / ".git").is_dir():
    run(["git", "-C", THREEDDFA_ROOT, "fetch", "origin"])
else:
    run(["git", "clone", "https://github.com/cleardusk/3DDFA_V2.git", THREEDDFA_ROOT])

run(["git", "-C", THREEDDFA_ROOT, "checkout", "--detach", THREEDDFA_COMMIT])
run(["bash", "build.sh"], cwd=THREEDDFA_ROOT)

actual_3ddfa_commit = subprocess.check_output(
    ["git", "-C", str(THREEDDFA_ROOT), "rev-parse", "HEAD"], text=True
).strip()
assert actual_3ddfa_commit == THREEDDFA_COMMIT
assert torch.cuda.is_available(), "Hãy đổi Colab runtime sang GPU trước khi chạy 3DDFA"
print("3DDFA commit:", actual_3ddfa_commit)
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
#@title 3A.4 Chạy hoặc tái sử dụng 3DDFA smoke 50 bona fide frames

# Official ONNX backend cần các package này.
run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "onnx",
        "onnxruntime-gpu",
    ]
)

import onnxruntime as ort

print(
    "ONNX Runtime providers:",
    ort.get_available_providers(),
)

depth_status = pd.read_csv(
    DEPTH_LEDGER,
    keep_default_na=False,
    dtype={"sample_id": str},
)

complete_live_before = int(
    (
        depth_status["status"]
        == "complete_bona_fide"
    ).sum()
)

failed_before = int(
    (
        depth_status["status"]
        == "failed_bona_fide"
    ).sum()
)

print(
    "Complete live trước smoke:",
    complete_live_before,
)
print(
    "Failures trước smoke:",
    failed_before,
)

SMOKE_TARGET = 50
smoke_remaining = max(
    0,
    SMOKE_TARGET - complete_live_before,
)

if smoke_remaining > 0:
    pending = pd.read_csv(DEPTH_PENDING)

    assert len(pending) >= smoke_remaining, (
        f"Không đủ pending samples: "
        f"need={smoke_remaining}, "
        f"available={len(pending)}"
    )

    run(
        [
            sys.executable,
            "scripts/run_3ddfa_worker.py",
            "--pending", DEPTH_PENDING,
            "--ledger", DEPTH_LEDGER,
            "--3ddfa-root", THREEDDFA_ROOT,
            "--onnx",
            "--mode", "gpu",
            "--limit", str(smoke_remaining),
            "--failure-report", DEPTH_FAILURES,
        ],
        cwd=REPO_ROOT,
    )

    save_depth_checkpoint()

else:
    print(
        "Đã có ít nhất 50 bona fide depth maps; "
        "tái sử dụng artifact và không chạy worker."
    )

depth_status = pd.read_csv(
    DEPTH_LEDGER,
    keep_default_na=False,
    dtype={"sample_id": str},
)

complete_live_after = int(
    (
        depth_status["status"]
        == "complete_bona_fide"
    ).sum()
)

failed_after = int(
    (
        depth_status["status"]
        == "failed_bona_fide"
    ).sum()
)

display(
    depth_status["status"]
    .value_counts()
    .rename("samples")
    .to_frame()
)

print(
    "Complete live sau smoke:",
    complete_live_after,
)
print(
    "Smoke failures:",
    failed_after,
)

assert complete_live_after >= SMOKE_TARGET, (
    "3DDFA smoke chưa đủ 50 bona fide maps"
)

assert failed_after == 0, (
    "Smoke có 3DDFA failure; cần review trước khi tiếp tục"
)

print("✓ 3DDFA smoke gate có đủ 50 bona fide maps")

In [ ]:
#@title 3A.5 Xuất contact sheet RGB–depth–mask–histogram
DEPTH_SMOKE_IMAGE = REPORTS_DIR / "casia-depth-smoke.png"
run(
    [
        sys.executable,
        "scripts/visualize_depth_targets.py",
        "--ledger", DEPTH_LEDGER,
        "--count", "12",
        "--output", DEPTH_SMOKE_IMAGE,
    ],
    cwd=REPO_ROOT,
)
display(DisplayImage(filename=str(DEPTH_SMOKE_IMAGE), width=1400))
print("Saved:", DEPTH_SMOKE_IMAGE)


### Cổng thủ công 3A

Chỉ duyệt nếu:

- live target có cấu trúc 3D khuôn mặt và không rỗng/không hằng;
- attack target bằng zero;
- RGB và target thẳng hàng hợp lý;
- không có tỷ lệ `failed_bona_fide` bất thường.

Sau khi kiểm tra ảnh, đổi biến trong cell kế tiếp thành `True`.


In [ ]:
#@title 3A.6 Xác nhận smoke gate
SMOKE_APPROVED = False  # Đổi thành True sau khi đã xem ảnh phía trên.
assert SMOKE_APPROVED, "Dừng ở đây: cần con người duyệt casia-depth-smoke.png"
print("✓ 3DDFA smoke gate được duyệt")


### 3B. Full pseudo-depth queue, retry và QA


In [ ]:
#@title 3B.1 Chạy full queue theo chunk và checkpoint lên Drive

assert SMOKE_APPROVED

CHUNK_SIZE = 250

while True:
    pending_count = len(
        pd.read_csv(DEPTH_PENDING)
    )

    if pending_count == 0:
        break

    print(
        f"\nPending trước chunk: "
        f"{pending_count}"
    )

    run(
        [
            sys.executable,
            "scripts/run_3ddfa_worker.py",
            "--pending", DEPTH_PENDING,
            "--ledger", DEPTH_LEDGER,
            "--3ddfa-root", THREEDDFA_ROOT,

            # Bắt buộc giữ cùng backend với smoke.
            "--onnx",

            "--mode", "gpu",
            "--limit", str(CHUNK_SIZE),
            "--failure-report", DEPTH_FAILURES,
        ],
        cwd=REPO_ROOT,
    )

    save_depth_checkpoint()

depth_status = pd.read_csv(
    DEPTH_LEDGER,
    keep_default_na=False,
    dtype={"sample_id": str},
)

display(
    depth_status.groupby(
        ["split", "label", "status"]
    )
    .size()
    .rename("samples")
    .to_frame()
)

In [ ]:
#@title 3B.2 Retry failure đúng một lần nếu có

depth_status = pd.read_csv(
    DEPTH_LEDGER,
    keep_default_na=False,
    dtype={"sample_id": str},
)

failed_before_retry = depth_status[
    depth_status["status"] == "failed_bona_fide"
]

print(
    "Failed trước retry:",
    len(failed_before_retry),
)

RETRY_MARKER = (
    DEPTH_BACKUP /
    "retry_attempted.json"
)

if (
    len(failed_before_retry)
    and not RETRY_MARKER.is_file()
):
    display(
        failed_before_retry[
            [
                "sample_id",
                "image_path",
                "last_error",
            ]
        ].head(30)
    )

    # Đưa riêng các failure bona fide trở lại pending.
    # Không truyền --failure-report ở bước này vì file đó
    # đang chứa chính các failure cần được retry.
    run(
        [
            sys.executable,
            "scripts/generate_depth.py",
            MANIFEST,
            "--data-root", DATA_ROOT,
            "--output-root", DEPTH_ROOT,
            "--retry-failed",
        ],
        cwd=REPO_ROOT,
    )

    retry_pending = len(
        pd.read_csv(DEPTH_PENDING)
    )

    print(
        "Samples được đưa trở lại queue:",
        retry_pending,
    )

    assert retry_pending > 0, (
        "Có failure nhưng không sample nào "
        "được đưa trở lại pending"
    )

    # Phải giữ nguyên official ONNX backend đã dùng
    # cho 50 smoke frames và toàn bộ full queue.
    run(
        [
            sys.executable,
            "scripts/run_3ddfa_worker.py",
            "--pending", DEPTH_PENDING,
            "--ledger", DEPTH_LEDGER,
            "--3ddfa-root", THREEDDFA_ROOT,
            "--onnx",
            "--mode", "gpu",
            "--failure-report", DEPTH_FAILURES,
        ],
        cwd=REPO_ROOT,
    )

    save_depth_checkpoint()

    # Chỉ ghi marker sau khi retry worker đã chạy xong.
    # Nếu worker bị crash, marker không được tạo và cell
    # vẫn có thể chạy lại.
    RETRY_MARKER.write_text(
        json.dumps(
            {
                "attempted_at_unix": time.time(),
                "attempted_failures": len(
                    failed_before_retry
                ),
                "backend": "onnx",
                "three_ddfa_commit": (
                    THREEDDFA_COMMIT
                ),
            },
            indent=2,
        ) + "\n",
        encoding="utf-8",
    )

    print(
        "Retry marker:",
        RETRY_MARKER,
    )

elif len(failed_before_retry):
    print(
        "Retry đã được thực hiện ở lần chạy trước; "
        "không tự động retry vô hạn:",
        RETRY_MARKER,
    )

else:
    print(
        "Không có 3DDFA failure; "
        "không cần retry."
    )

# Đọc lại trạng thái cuối cùng.
depth_status = pd.read_csv(
    DEPTH_LEDGER,
    keep_default_na=False,
    dtype={"sample_id": str},
)

pending_count = int(
    (
        depth_status["status"]
        == "pending_3ddfa"
    ).sum()
)

failed_count = int(
    (
        depth_status["status"]
        == "failed_bona_fide"
    ).sum()
)

live_count = int(
    (
        depth_status["status"]
        == "complete_bona_fide"
    ).sum()
)

attack_count = int(
    (
        depth_status["status"]
        == "complete_attack_zero"
    ).sum()
)

failure_rate = (
    failed_count /
    max(1, live_count + failed_count)
)

print(f"Pending: {pending_count}")
print(f"Complete live: {live_count}")
print(f"Complete attack zero: {attack_count}")
print(f"Persistent failures: {failed_count}")
print(
    "3DDFA failure rate: "
    f"{failure_rate:.4%}"
)

display(
    depth_status["status"]
    .value_counts()
    .rename("samples")
    .to_frame()
)

assert pending_count == 0, (
    "Queue vẫn còn pending samples"
)

assert failed_count == 0, (
    "Persistent 3DDFA failures: "
    "dừng và review, không tự loại sample"
)

assert live_count == 3000, (
    f"Expected 3000 complete live maps, "
    f"found {live_count}"
)

assert attack_count == 9000, (
    f"Expected 9000 attack-zero maps, "
    f"found {attack_count}"
)

print("✓ Full pseudo-depth queue hoàn tất")

In [ ]:
#@title 3B.3 Materialize manifest, verify provenance và validate
run(
    [
        sys.executable,
        "scripts/materialize_depth_manifest.py",
        MANIFEST,
        "--ledger", DEPTH_LEDGER,
        "--data-root", DATA_ROOT,
        "--output", DEPTH_MANIFEST,
    ],
    cwd=REPO_ROOT,
)
run(
    [
        sys.executable,
        "scripts/verify_depth_provenance.py",
        "--source", MANIFEST,
        "--ledger", DEPTH_LEDGER,
        "--derived", DEPTH_MANIFEST,
    ],
    cwd=REPO_ROOT,
)
run(
    [
        sys.executable,
        "scripts/validate_manifest.py",
        DEPTH_MANIFEST,
        "--data-root", DATA_ROOT,
        "--subject-disjoint",
        "--require-depth",
    ],
    cwd=REPO_ROOT,
)


In [ ]:
#@title 3B.4 Full depth audit và lưu bằng chứng lên Drive
DEPTH_QA = REPORTS_DIR / "casia-depth-qa.json"
run(
    [
        sys.executable,
        "scripts/audit_depth.py",
        DEPTH_MANIFEST,
        "--data-root", DATA_ROOT,
        "--report", DEPTH_QA,
    ],
    cwd=REPO_ROOT,
)

qa = json.loads(DEPTH_QA.read_text())
display(pd.json_normalize(qa))
assert qa.get("valid") is True, "Depth QA chưa hợp lệ; không được train E1"

evidence = [
    DEPTH_LEDGER,
    Path(str(DEPTH_LEDGER) + ".worker.json"),
    DEPTH_FAILURES,
    DEPTH_MANIFEST,
    Path(str(DEPTH_MANIFEST) + ".provenance.json"),
]
for source in evidence:
    if source.is_file():
        shutil.copy2(source, REPORTS_DIR / source.name)
save_depth_checkpoint()

print("✓ Depth QA valid")
print("Report:", DEPTH_QA)


### 3B.5 Face-crop demo dùng vùng mặt của 3DDFA

Artifact này minh họa bước phát hiện/crop mặt còn thiếu ở Lượt 1. Bounding box được
suy ra từ vùng depth khác zero do FaceBoxes + 3DDFA tái dựng. Crop chỉ dùng để minh
họa preprocessing; E0 và E1 CASIA đã khóa vẫn dùng cùng full-frame convention và
không được train lại bằng các crop này.


In [ ]:
#@title 3B.5 Tạo face-crop demo từ vùng 3DDFA
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

depth_status = pd.read_csv(
    DEPTH_LEDGER,
    keep_default_na=False,
    dtype={"sample_id": str},
)

live_rows = depth_status[
    depth_status["status"] == "complete_bona_fide"
].sample(n=8, random_state=42)

crop_cases = []

for row in live_rows.itertuples(index=False):
    rgb = Image.open(Path(str(row.image_path))).convert("RGB")
    depth = np.asarray(
        np.load(Path(str(row.output_path)), allow_pickle=False),
        dtype=np.float32,
    )
    mask = np.abs(depth) > 1e-6
    ys, xs = np.where(mask)
    assert len(xs), f"Depth mask rỗng: {row.sample_id}"

    image_width, image_height = rgb.size
    depth_height, depth_width = depth.shape
    x0 = int(xs.min() / depth_width * image_width)
    x1 = int((xs.max() + 1) / depth_width * image_width)
    y0 = int(ys.min() / depth_height * image_height)
    y1 = int((ys.max() + 1) / depth_height * image_height)

    padding_x = int((x1 - x0) * 0.20)
    padding_y = int((y1 - y0) * 0.20)
    x0 = max(0, x0 - padding_x)
    y0 = max(0, y0 - padding_y)
    x1 = min(image_width, x1 + padding_x)
    y1 = min(image_height, y1 + padding_y)
    assert x1 > x0 and y1 > y0, f"Invalid crop: {row.sample_id}"

    boxed = rgb.copy()
    ImageDraw.Draw(boxed).rectangle(
        (x0, y0, x1, y1),
        outline="lime",
        width=max(2, image_width // 150),
    )
    crop_cases.append(
        {
            "sample_id": row.sample_id,
            "boxed": boxed,
            "crop": rgb.crop((x0, y0, x1, y1)),
        }
    )

fig, axes = plt.subplots(4, 4, figsize=(14, 14))
for case_index, case in enumerate(crop_cases):
    row_index = case_index // 2
    column_index = (case_index % 2) * 2
    axes[row_index, column_index].imshow(case["boxed"])
    axes[row_index, column_index].set_title(f'{case["sample_id"]} — detected region')
    axes[row_index, column_index].axis("off")
    axes[row_index, column_index + 1].imshow(case["crop"])
    axes[row_index, column_index + 1].set_title("face crop demo")
    axes[row_index, column_index + 1].axis("off")

fig.tight_layout()
CROP_DEMO = REPORTS_DIR / "casia-3ddfa-face-crop-demo.png"
fig.savefig(CROP_DEMO, dpi=180, bbox_inches="tight")
plt.show()
print("Saved:", CROP_DEMO)


In [ ]:
#@title Sanity test ContrastiveDepthLoss trên A100

import deepface_pad.losses as losses_module

losses_module = importlib.reload(
    losses_module
)

prediction = torch.rand(
    2,
    1,
    32,
    32,
    device="cuda",
    requires_grad=True,
)

# Cố ý để target ở CPU để kiểm tra việc tự đồng bộ.
target = torch.rand(
    2,
    1,
    32,
    32,
    device="cpu",
)

test_loss = losses_module.depth_loss(
    prediction,
    target,
)

test_loss.backward()

print("Loss:", float(test_loss.detach()))
print("Prediction device:", prediction.device)
print("Gradient device:", prediction.grad.device)
print("Finite:", torch.isfinite(test_loss).item())

assert prediction.grad is not None
assert prediction.grad.device.type == "cuda"
assert torch.isfinite(test_loss)

print("✓ ContrastiveDepthLoss hoạt động trên GPU")

### 3C. E1 smoke


In [ ]:
#@title 3C.1 Chạy hoặc tái sử dụng E1 smoke 1 epoch
assert qa.get("valid") is True

smoke_runs = [
    path for path in RUNS_DIR.glob("CASIA_E1_SMOKE_seed42_*")
    if (path / "best.ckpt").is_file() and (path / "metrics.json").is_file()
    and (path / "config.yaml").read_text() == (REPO_ROOT / "configs/casia_e1_smoke.yaml").read_text()
]
if smoke_runs:
    print("Tái sử dụng E1 smoke run đã hoàn tất; không tiêu tốn GPU để train lại.")
else:
    run([sys.executable, "scripts/run_experiment.py", "configs/casia_e1_smoke.yaml"], cwd=REPO_ROOT)
    smoke_runs = [
        path for path in RUNS_DIR.glob("CASIA_E1_SMOKE_seed42_*")
        if (path / "best.ckpt").is_file() and (path / "metrics.json").is_file()
        and (path / "config.yaml").read_text() == (REPO_ROOT / "configs/casia_e1_smoke.yaml").read_text()
    ]
smoke_runs = sorted(smoke_runs, key=lambda path: path.stat().st_mtime)
assert smoke_runs
E1_SMOKE_RUN = smoke_runs[-1]
print("E1 smoke run:", E1_SMOKE_RUN)


In [ ]:
#@title 3C.2 Kiểm tra loss, metric và checkpoint E1 smoke
smoke_log = pd.read_csv(E1_SMOKE_RUN / "train_log.csv")
smoke_metrics = json.loads((E1_SMOKE_RUN / "metrics.json").read_text())
display(smoke_log)
display(pd.DataFrame([smoke_metrics], index=["E1 smoke validation"]))

assert np.isfinite(smoke_log["train_loss"]).all()
assert np.isfinite(smoke_log["val_loss"]).all()
assert (E1_SMOKE_RUN / "best.ckpt").is_file()
assert (E1_SMOKE_RUN / "depth_input_snapshot.json").is_file()
print("✓ Loss hữu hạn, best checkpoint và depth snapshot tồn tại")


In [ ]:
#@title 3C.3 Xuất predicted-depth cases của E1 smoke
E1_SMOKE_CASES = REPORTS_DIR / f"{E1_SMOKE_RUN.name}-depth-cases"
run(
    [
        sys.executable,
        "scripts/visualize_depth_cases.py",
        "--run-dir", E1_SMOKE_RUN,
        "--split", "val",
        "--count", "10",
        "--output-dir", E1_SMOKE_CASES,
    ],
    cwd=REPO_ROOT,
)

case_images = sorted(E1_SMOKE_CASES.glob("*.png"))
assert len(case_images) >= 10
for path in case_images[:5]:
    display(DisplayImage(filename=str(path), width=1200))
print("All cases:", E1_SMOKE_CASES)


### Cổng thủ công 3C

Chỉ duyệt E1 smoke nếu loss hữu hạn, checkpoint tồn tại và predicted depth không sụp thành NaN/map hằng vô nghĩa. Đổi biến dưới đây thành `True` sau khi xem kết quả.


In [ ]:
#@title 3C.4 Xác nhận E1 smoke gate
E1_SMOKE_APPROVED = False  # Đổi thành True sau khi review loss và depth cases.
assert E1_SMOKE_APPROVED, "Dừng ở đây: cần duyệt E1 smoke trước khi chạy 30 epochs"
print("✓ E1 smoke gate được duyệt")


### 3D. E1 full, validation evidence và locked test


In [ ]:
#@title 3D.1 Chạy hoặc tái sử dụng E1 full 30 epochs
assert E1_SMOKE_APPROVED

full_runs = [
    path for path in RUNS_DIR.glob("CASIA_E1_CDCN_seed42_*")
    if (path / "best.ckpt").is_file() and (path / "metrics.json").is_file()
    and (path / "config.yaml").read_text() == (REPO_ROOT / "configs/casia_e1_cdcn.yaml").read_text()
]
if full_runs:
    print("Tái sử dụng E1 full run đã hoàn tất; không train lại 30 epochs.")
else:
    run([sys.executable, "scripts/run_experiment.py", "configs/casia_e1_cdcn.yaml"], cwd=REPO_ROOT)
    full_runs = [
        path for path in RUNS_DIR.glob("CASIA_E1_CDCN_seed42_*")
        if (path / "best.ckpt").is_file() and (path / "metrics.json").is_file()
        and (path / "config.yaml").read_text() == (REPO_ROOT / "configs/casia_e1_cdcn.yaml").read_text()
    ]
full_runs = sorted(full_runs, key=lambda path: path.stat().st_mtime)
assert full_runs
E1_RUN = full_runs[-1]
print("E1 full run:", E1_RUN)


In [ ]:
#@title 3D.2 Kiểm tra training log và validation metrics E1
e1_log = pd.read_csv(E1_RUN / "train_log.csv")
e1_val_metrics = json.loads((E1_RUN / "metrics.json").read_text())
e1_threshold = json.loads((E1_RUN / "threshold.json").read_text())

display(e1_log)
display(pd.DataFrame([e1_val_metrics], index=["E1 validation"]))
print("Threshold record:", e1_threshold)

assert np.isfinite(e1_log["train_loss"]).all()
assert np.isfinite(e1_log["val_loss"]).all()
assert e1_threshold.get("source") == "validation"
assert (E1_RUN / "best.ckpt").is_file()
assert (E1_RUN / "depth_input_snapshot.json").is_file()


In [ ]:
#@title 3D.3 Xuất ít nhất 10 validation depth cases
E1_CASES = REPORTS_DIR / f"{E1_RUN.name}-depth-cases"
run(
    [
        sys.executable,
        "scripts/visualize_depth_cases.py",
        "--run-dir", E1_RUN,
        "--split", "val",
        "--count", "12",
        "--output-dir", E1_CASES,
    ],
    cwd=REPO_ROOT,
)

case_images = sorted(E1_CASES.glob("*.png"))
assert len(case_images) >= 10
for path in case_images[:6]:
    display(DisplayImage(filename=str(path), width=1200))
print("All validation cases:", E1_CASES)


### Locked-test gate

Trước cell tiếp theo, xác nhận rằng:

- config E1 full sẽ không thay đổi nữa;
- checkpoint đã chọn bằng validation;
- threshold có `source: validation`;
- không sử dụng test để chọn epoch, kiến trúc hoặc hyperparameter.

Nếu `test_metrics.json` đã tồn tại, cell chỉ đọc lại và không score lần nữa.


In [ ]:
#@title 3D.4 Score locked test đúng một lần
RUN_LOCKED_TEST = False  # Chỉ đổi True khi model/config/threshold đã khóa.

test_metrics_path = E1_RUN / "test_metrics.json"
if test_metrics_path.is_file():
    print("Locked test đã tồn tại; không chạy lại:", test_metrics_path)
else:
    assert RUN_LOCKED_TEST, "Locked test chưa được cấp phép. Review validation trước."
    run(
        [
            sys.executable,
            "scripts/score_checkpoint.py",
            "--run-dir", E1_RUN,
            "--split", "test",
        ],
        cwd=REPO_ROOT,
    )

E1_TEST_METRICS = json.loads(test_metrics_path.read_text())
display(pd.DataFrame([E1_TEST_METRICS], index=["E1 locked test"]))


In [ ]:
#@title 3D.5 Error analysis E1 locked test
e1_scores = pd.read_csv(E1_RUN / "test_scores.csv").reset_index(drop=True)
metadata = (
    manifest[["video_id", "subject_id", "label", "attack_type", "quality"]]
    .drop_duplicates("video_id")
    .reset_index(drop=True)
)
e1_analysis = e1_scores.merge(metadata, on=["video_id", "label"], how="left")
e1_analysis["prediction"] = (
    e1_analysis["score"] >= float(E1_TEST_METRICS["threshold"])
).astype(int)
e1_analysis["correct"] = e1_analysis["prediction"] == e1_analysis["label"]

display(
    e1_analysis.groupby(["label", "attack_type", "quality"], dropna=False).agg(
        videos=("video_id", "count"),
        errors=("correct", lambda values: (~values).sum()),
        mean_score=("score", "mean"),
        min_score=("score", "min"),
        max_score=("score", "max"),
    )
)
print("Misclassified bona fide")
display(e1_analysis[(e1_analysis["label"] == 1) & (~e1_analysis["correct"])].sort_values("score"))
print("Highest-scoring attacks")
display(e1_analysis[e1_analysis["label"] == 0].nlargest(15, "score"))

e1_analysis.to_csv(REPORTS_DIR / f"{E1_RUN.name}-test-error-analysis.csv", index=False)


## 4. So sánh E0–E1 và đóng gói bằng chứng


In [ ]:
#@title 4.0 Nạp frozen artifacts để chỉ sinh figure/report
# Cell này không train, không score test và không thay đổi threshold.
L1_BATCH = REPORTS_DIR / "casia-val-batch.png"
DEPTH_SMOKE_IMAGE = REPORTS_DIR / "casia-depth-smoke.png"
DEPTH_QA = REPORTS_DIR / "casia-depth-qa.json"

E0_RUN = RUNS_DIR / "CASIA_E0_BCE_5E_seed42_20260920T082941Z"
assert E0_RUN.is_dir(), f"Không tìm thấy E0 frozen run: {E0_RUN}"

e0_test_metrics_path = E0_RUN / "test_metrics.json"
if e0_test_metrics_path.is_file():
    E0_TEST_METRICS = json.loads(e0_test_metrics_path.read_text())
else:
    E0_TEST_METRICS = {
        "threshold": 0.9900876432657242,
        "apcer": 0.0,
        "bpcer": 0.1111111111111111,
        "acer": 0.05555555555555555,
        "eer": 0.024074074074074074,
        "auc": 0.9979835390946502,
    }

e1_runs = sorted(
    [
        path
        for path in RUNS_DIR.glob("CASIA_E1_CDCN_seed42_*")
        if (path / "best.ckpt").is_file()
        and (path / "metrics.json").is_file()
        and (path / "test_metrics.json").is_file()
        and (path / "test_scores.csv").is_file()
    ],
    key=lambda path: path.stat().st_mtime,
)
assert e1_runs, "Không tìm thấy frozen E1 full run trên Drive"
E1_RUN = e1_runs[-1]
E1_TEST_METRICS = json.loads((E1_RUN / "test_metrics.json").read_text())
E1_CASES = REPORTS_DIR / f"{E1_RUN.name}-depth-cases"

smoke_runs = sorted(
    [
        path
        for path in RUNS_DIR.glob("CASIA_E1_SMOKE_seed42_*")
        if (path / "best.ckpt").is_file()
        and (path / "metrics.json").is_file()
    ],
    key=lambda path: path.stat().st_mtime,
)
assert smoke_runs, "Không tìm thấy frozen E1 smoke run trên Drive"
E1_SMOKE_RUN = smoke_runs[-1]

print("E0 frozen:", E0_RUN)
print("E1 smoke frozen:", E1_SMOKE_RUN)
print("E1 full frozen:", E1_RUN)
print("Không có training hoặc test scoring nào được thực hiện.")


In [ ]:
#@title 4.1 Bảng so sánh E0–E1 cùng protocol
comparison = pd.DataFrame(
    {"E0_MobileNetV3": E0_TEST_METRICS, "E1_CDCN": E1_TEST_METRICS}
).T[["threshold", "apcer", "bpcer", "acer", "eer", "auc"]]

comparison["acer_delta_vs_E0"] = comparison["acer"] - comparison.loc["E0_MobileNetV3", "acer"]
display(
    comparison.style.format(
        {
            "threshold": "{:.8f}",
            "apcer": "{:.4%}",
            "bpcer": "{:.4%}",
            "acer": "{:.4%}",
            "eer": "{:.4%}",
            "auc": "{:.6f}",
            "acer_delta_vs_E0": "{:+.4%}",
        }
    )
)

comparison_path = REPORTS_DIR / f"E0-vs-{E1_RUN.name}.csv"
comparison.to_csv(comparison_path)
print("Saved:", comparison_path)


### 4.2 Figure đóng gói cho báo cáo

Các figure dưới đây được sinh trực tiếp từ log và video-level raw scores đã khóa.
Chúng chỉ phục vụ báo cáo; tuyệt đối không dùng test distribution để đổi threshold.


In [ ]:
#@title 4.2 Sinh training curves E0–E1
import matplotlib.pyplot as plt

curve_runs = [
    ("E0 MobileNetV3", E0_RUN, "#2878B5"),
    ("E1 CDCN", E1_RUN, "#D95319"),
]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for axis, (name, run_dir, color) in zip(axes, curve_runs):
    log = pd.read_csv(run_dir / "train_log.csv")
    axis.plot(log["epoch"], log["train_loss"], label="train", color=color)
    axis.plot(
        log["epoch"],
        log["val_loss"],
        label="validation",
        color="black",
        linestyle="--",
    )
    best_index = log["val_loss"].idxmin()
    best_epoch = int(log.loc[best_index, "epoch"])
    best_loss = float(log.loc[best_index, "val_loss"])
    axis.scatter(
        [best_epoch],
        [best_loss],
        color="red",
        zorder=3,
        label=f"best epoch {best_epoch}",
    )
    axis.set_title(name)
    axis.set_xlabel("Epoch")
    axis.set_ylabel("Loss")
    axis.grid(alpha=0.25)
    axis.legend()

fig.tight_layout()
TRAINING_FIGURE = REPORTS_DIR / "e0-e1-training-curves.png"
fig.savefig(TRAINING_FIGURE, dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", TRAINING_FIGURE)


In [ ]:
#@title 4.3 Sinh test score distributions E0–E1
score_runs = [
    ("E0 MobileNetV3", E0_RUN, E0_TEST_METRICS["threshold"]),
    ("E1 CDCN", E1_RUN, E1_TEST_METRICS["threshold"]),
]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for axis, (name, run_dir, threshold) in zip(axes, score_runs):
    scores = pd.read_csv(run_dir / "test_scores.csv")
    attack = scores.loc[scores["label"] == 0, "score"]
    live = scores.loc[scores["label"] == 1, "score"]
    axis.hist(
        attack,
        bins=30,
        alpha=0.65,
        label=f"attack (n={len(attack)})",
        color="#D95319",
    )
    axis.hist(
        live,
        bins=30,
        alpha=0.65,
        label=f"bona fide (n={len(live)})",
        color="#2878B5",
    )
    axis.axvline(
        threshold,
        color="black",
        linestyle="--",
        label=f"threshold={threshold:.4f}",
    )
    axis.set_title(name)
    axis.set_xlabel("Video-level live score")
    axis.set_ylabel("Number of videos")
    axis.grid(alpha=0.20)
    axis.legend()

fig.tight_layout()
SCORE_FIGURE = REPORTS_DIR / "e0-e1-test-score-distributions.png"
fig.savefig(SCORE_FIGURE, dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", SCORE_FIGURE)


In [ ]:
#@title 4.4 Kiểm tra artifact cuối cùng
required_e1 = [
    "config.yaml",
    "environment.txt",
    "manifest_checksum.json",
    "train_log.csv",
    "best.ckpt",
    "val_frame_scores.csv",
    "val_scores.csv",
    "threshold.json",
    "metrics.json",
    "depth_input_snapshot.json",
    "test_frame_scores.csv",
    "test_scores.csv",
    "test_metrics.json",
]

artifact_table = pd.DataFrame(
    [
        {
            "artifact": name,
            "exists": (E1_RUN / name).is_file(),
            "path": str(E1_RUN / name),
        }
        for name in required_e1
    ]
)
display(artifact_table)
assert artifact_table["exists"].all(), "E1 run vẫn thiếu artifact bắt buộc"

print("✓ Lượt 1 artifact:", L1_BATCH)
print("✓ E0 frozen run:", E0_RUN)
print("✓ Depth smoke:", DEPTH_SMOKE_IMAGE)
print("✓ Depth QA:", DEPTH_QA)
print("✓ E1 smoke run:", E1_SMOKE_RUN)
print("✓ E1 full run:", E1_RUN)
print("✓ E1 depth cases:", E1_CASES)
print("✓ E0–E1 comparison:", comparison_path)

required_reports = [
    L1_BATCH,
    DEPTH_SMOKE_IMAGE,
    DEPTH_QA,
    CROP_DEMO,
    TRAINING_FIGURE,
    SCORE_FIGURE,
    comparison_path,
]
missing_reports = [str(path) for path in required_reports if not Path(path).exists()]
assert not missing_reports, f"Thiếu report artifacts: {missing_reports}"
print("✓ Crop demo:", CROP_DEMO)
print("✓ Training curves:", TRAINING_FIGURE)
print("✓ Score distributions:", SCORE_FIGURE)
print("✓ Artifact Lượt 1–3 đã đầy đủ")


## Hoàn tất Lượt 1–3

Khi cell artifact cuối cùng pass, dữ kiện cần thiết cho phần báo cáo giữa kỳ đã đủ:

- protocol và leakage checks;
- E0 frozen result;
- depth smoke image, queue failure rate, provenance và QA;
- E1 smoke/full logs, validation threshold và locked test metrics;
- ít nhất 10 predicted-depth cases;
- bảng E0–E1 cùng protocol.

OULU-NPU và Replay-Attack vẫn là công việc tiếp theo sau khi quyền dataset được duyệt. Không dùng kết quả CASIA tạm thời như bằng chứng cross-dataset.
